In [ ]:
from pathlib import Path
import pandas as pd

import ixmp4
import pyam
import nomenclature

In [ ]:
platform = ixmp4.Platform("scenariocompass-transfer")

In [ ]:
project_id = "DAC"
project_name = "DAC"
project_meta_indicator = "DAC (MERGE-ETL)"

In [ ]:
ar6_db = pyam.iiasa.Connection("ar6-public")

In [ ]:
runs = ar6_db.properties().reset_index()

In [ ]:
runs[[i.startswith(project_id) for i in runs.scenario.values]].model.unique()

In [ ]:
meta = pd.read_excel("raw/AR6/AR6_Scenarios_Database_metadata_indicators_v1.1.xlsx", sheet_name="meta_Ch3vetted_withclimate")

In [ ]:
meta = meta[['Model', 'Scenario', 'Category_subset', 'Literature Reference (if applicable)']]
meta.rename(
    columns={
        "Model": "model",
        "Scenario": "scenario",
        "Category_subset": "Climate Assessment|AR6|Category [ID]",
        "Literature Reference (if applicable)": "manuscript",
    },
    inplace=True,
)

In [ ]:
ar6_models = set(meta.model.unique()).intersection(runs[[i.startswith(project_id) for i in runs.scenario.values]].model.unique())

In [ ]:
ar6_models

In [ ]:
#ar6_models = ["REMIND-MAgPIE 2.1-4.2"]

In [ ]:
#model = "EPPA 6"
#scenario = "*"

#for scenario in runs[runs.model == "IMAGE 3.2"].scenario:
#df = ar6_db.query(model=model, scenario=scenario)
#df.to_excel(f"raw/AR6/{project_name}_{model.replace(' ', '_').replace('/', '_')}_{scenario.replace('*', '').replace(' ', '_').replace('/', '_')}.xlsx")

In [ ]:
for model in ar6_models:
    df = ar6_db.query(model=model, scenario=f"{project_id}*")
    df.to_excel(f"raw/AR6/{project_name}_{model.replace(' ', '_').replace('/', '_')}.xlsx")

In [ ]:
df = pyam.concat(
    [
        i for i in list(Path("raw/AR6/").iterdir()) if i.name.startswith(project_name)
    ]
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                i.replace("2?C", "2°C")
            )
            for i in df.scenario if i.startswith("NGFS2")
        ]
    ),
    inplace=True,
)

In [ ]:
df = pyam.IamDataFrame(df.data, meta=meta)

In [ ]:
df.meta["Climate Assessment|AR6|Category [ID]"].unique()

In [ ]:
df.meta[df.meta["Climate Assessment|AR6|Category [ID]"].isna()]

In [ ]:
df.meta.replace(
    {
        'C1a_NZGHGs': "C1a",
        "C1b_+veGHGs": "C1b",
        'C3y_+veGHGs': "C3y",
        'C3x_NZGHGs': "C3x",
    },
    inplace=True,
)

In [ ]:
# this line implictly removes all non-categorized scenarios
df.filter(
    **{"Climate Assessment|AR6|Category [ID]": df.meta["Climate Assessment|AR6|Category [ID]"].unique()},
    inplace=True,
)

In [ ]:
len(df.meta)

In [ ]:
df.meta['manuscript'].unique()

In [ ]:
df.meta

In [ ]:
for full, doi, name in [
    ("https://doi.org/10.1016/j.gloenvcha.2016.05.009", "10.1016/j.gloenvcha.2016.05.009", "Riahi et al. (2017)"),
    ("https://doi.org/10.1038/s41558-018-0091-3", "10.1038/s41558-018-0091-3", "Rogelj et al. (2018)"),
    ("https://doi.org/10.1038/s41467-021-26509-z", "10.1038/s41467-021-26509-z", "Ou et al. (2021)"),
    ("https://doi.org/10.1038/s41467-021-26595-z", "10.1038/s41467-021-26595-z", "van Soest et. al (2021)"),
    ("https://doi.org/10.1038/s41560-018-0179-z", "10.1038/s41560-018-0179-z", "McCollum et al. (2018)"),
    ("https://doi.org/10.1007/s10584-020-02837-9", "10.1007/s10584-020-02837-9", "Schaeffer et al. (2020)"),
    (" https://doi.org/10.1038/s41560-018-0179-z; https://doi.org/10.1038/s41467-020-15414-6 ", "10.1038/s41467-020-15414-6", "Roelfsema et al. (2020)"),
    ("https://doi.org/10.1038/s41560-018-0179-z; https://doi.org/10.1038/s41467-020-15414-6", "10.1038/s41467-020-15414-6", "Roelfsema et al. (2020)"),
    ("https://doi.org/10.1088/1748-9326/ab9966 ", "10.1088/1748-9326/ab9966", "Fujimori et al. (2020)"),
    ("https://doi.org/10.1038/s41467-021-22211-2", "10.1038/s41467-021-22211-2", "Strefler, Kriegler et al. (2021)"),
    ("https://doi.org/10.1088/1748-9326/ac0a11", "10.1088/1748-9326/ac0a11", "Strefler, Bauer et al. (2021)"),
    (
        "https://doi.org/10.1016/j.energy.2021.121547; https://doi.org/10.1007/s10584-013-0906-1; https://doi.org/10.31223/X5CG92  ",
        "10.31223/X5CG92",
        "van Vuuren et al. (2021)"
    ),
    ("https://doi.org/10.1038/s41560-018-0172-6", "10.1038/s41560-018-0172-6", "Grubler et al. (2018)"),
    ("https://doi.org/10.1038/s41560-021-00904-8", "10.1038/s41560-021-00904-8", "Kikstra et al. (2021)"), 
    ("https://doi.org/10.1088/1748-9326/aac4f1", "10.1088/1748-9326/aac4f1", "Kriegler et al. (2018)"),
    ("https://doi.org/10.1088/1748-9326/ac27ce", "10.1088/1748-9326/ac27ce", "Schultes et al. (2021)"),
    ("https://doi.org/10.1016/j.energy.2013.01.016", "10.1088/1748-9326/ab9966", "Fujimori et al. (2020)"),
    ("https://doi.org/10.1038/s41560-021-00937-z ", "10.1038/s41560-021-00937-z", "Luderer et al. (2021)"),
    ("https://doi.org/10.1038/s41558-021-01098-3", "10.1038/s41558-021-01098-3", "Soergel et al. (2021)"),
    ("https://doi.org/10.5194/gmd-2021-85 ", "10.5194/gmd-2021-85", "Baumstark et al. (2021)"),
    ("https://doi.org/10.1016/j.energy.2020.119253", "10.1016/j.energy.2020.119253", "Giannousakis et al. (2021)"),
    ("https://doi.org/10.1016/j.trd.2021.103005 ", "10.1016/j.trd.2021.103005", "Rottoli et al. (2021)"),
    ("https://doi.org/10.1088/1748-9326/abdf07", "10.1088/1748-9326/abdf07", "Levesque et al. (2021)"),
    (
        "https://doi.org/10.1007/s10584-020-02945-6; https://doi.org/10.1007/s10584-018-2226-y",
        "10.1007/s10584-018-2226-y",
        "Bauer et al. (2020)",
    ),
    ("https://doi.org/10.1088/1748-9326/aac0c1", "10.1088/1748-9326/aac0c1", "Holz et al. (2018)"),
    ('https://doi.org/10.1007/s10584-020-02938-5', "10.1007/s10584-020-02938-5", "Smith et al. (2020)"),
    ('https://doi.org/10.1007/s10584-019-02437-2', "10.1007/s10584-019-02437-2", "Harmsen et al. (2020)"),
    ('https://doi.org/10.1038/s41558-018-0198-6', "10.1038/s41558-018-0198-6", "Luderer et al. (2018)"),
    ('https://doi.org/10.1088/1748-9326/aab53e', "10.1088/1748-9326/aab53e", "Vrontisi et al. (2018)"),
    ('https://doi.org/10.1038/s41558-018-0198-6 ', "10.1038/s41558-018-0198-6", "Luderer et al. (2018)"),
    ('https://doi.org/10.1038/s41586-020-2982-5', "10.1038/s41586-020-2982-5", "Bauer et al. (2020)"),
    (
        'NGFS Climate Scenarios for central banks and supervisors, NGFS June 2020. https://www.ngfs.net/sites/default/files/medias/documents/820184_ngfs_scenarios_final_version_v6.pdf',
        'n/a',
        "NGFS (2020)"
    ),
    (
       'NGFS Climate Scenarios for central banks and supervisors, NGFS June 2021. https://www.ngfs.net/sites/default/files/media/2021/08/27/ngfs_climate_scenarios_phase2_june2021.pdf',
       'n/a',
       "NGFS (2021)"
    ),
    ("https://doi.org/10.1038/s41558-021-01206-3", "10.1038/s41558-021-01206-3", "Sognnaes et al. (2021)"),
    ('https://doi.org/10.1088/1748-9326/aac3ec', "10.1088/1748-9326/aac3ec", "Bertram et al. (2018)"),
    ('https://doi.org/10.1007/s10584-017-2051-8', '10.1007/s10584-017-2051-8', "Marcucci et al. (2017)"),
    ('https://doi.org/10.1088/1748-9326/ab3cc9', '10.1088/1748-9326/ab3cc9', "Emmerling et al. (2019)"),
]:
    index = df.filter(manuscript=full).index
    df.set_meta(meta=doi, name="Scientific Manuscript (DOI)", index=index)
    df.set_meta(meta=name, name="Scientific Manuscript (Citation)", index=index)


In [ ]:
#df.set_meta(meta="n/a" , name="Scientific Manuscript (DOI)")
#df.set_meta(meta="NGFS (2021)", name="Scientific Manuscript (Citation)")

In [ ]:
df.meta["Scientific Manuscript (Citation)"].unique()

In [ ]:
df.meta.drop(columns="Climate Assessment|AR6|Category [ID]", inplace=True)

In [ ]:
df.meta.drop(columns="manuscript", inplace=True)

In [ ]:
df.meta

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                i
                .replace("_", "-")
                .replace("Ref", "Reference")
            )
            for i in df.scenario if i.startswith("DISCRATE")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(scenario={"DAC2_66": "DAC-2°C-66%"}, inplace=True)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                i.replace("2C", "2°C").replace("1p5C", "1.5°C")
                .replace("_", "-")
                .replace("REF", "Reference")
                .replace("Def", "Default")
                .replace("regul", "Regulation")
                .replace("early", "Early-Action")
                .replace("lifesty", "Lifestyle")
                .replace("Sust", "Sustainable")
            )
            for i in df.scenario if i.startswith("SMP")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                   "ParisReinforce-" + i[3:].replace("_", "-").replace("base", "Base").replace("CP", "Current-Policies")
                )
            )
            for i in df.scenario if i.startswith("PR_")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                   "NGFS Phase 2-" + i[6:].replace("- IPD-95th", "[IPD 95th]").replace("- IPD-median", "[IPD Median]")
                )
            )
            for i in df.scenario if i.startswith("NGFS2")
        ]
    ),
    inplace=True,
)

In [ ]:
df.scenario

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                   i.replace("_", "-").replace("No-policy-baseline", "NoPolicy-Baseline").replace("_def", "")
                )
            )
            for i in df.scenario if i.startswith("Diff")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                   i.replace("C", "°C")
                    .replace("WB", "well-below-")
                    .replace("Med", "medium-")
                    .replace("_", "-")
                    .replace("Price", "Price ")
                    
                )
            )
            for i in df.scenario if i.startswith("ADVANCE")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    i.replace("_", "-")
                    .replace("ClimPolicy", "ClimatePolicy")
                    .replace("faster", "Faster")
                )
            )
            for i in df.scenario if i.startswith("EMF30")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    i.replace("DeepElec", "DeepElectrification")
                    .replace("_", "-")
                    .replace("Base", "Baseline")
                    .replace("Npi", "NPi")
                    .replace("def", "Default")
                    .replace(" HighRE", "HighRenewables")
                    .replace("Budg", "")
                )
            )
            for i in df.scenario if i.startswith("DeepElec")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                i.replace("Budg", "")
                .replace("Base", "Baseline")
               .replace("-EG", "-Efficiency")
            )
            for i in df.scenario if i.startswith("BEG")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                i.replace("_Budg", "-")
                .replace("Conv", "Conventional")
                .replace("Syn", "-SynFuel")
                .replace("LowD", "LowDemand")
                .replace("ElecPush", "Electrification")
                .replace("H2Push", "H2")
                .replace("_", "-")
            )
            for i in df.scenario if i.startswith("Transport")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    model={"EPPA 6": "EPPA 6"},
    scenario=dict(
        [
            (
                i,
                "MIT-JP-" + i.replace("1.5C", "1.5°C")
                .replace("2C", "2°C")
                .replace("Ref", "Reference")
                .replace("Paris", "Paris-")
                .replace("Now", "-Now")
                .replace("_", "-")
            )
            for i in df.filter(model="EPPA 6").scenario
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                i.replace("B1100", "1100")
            )
            for i in df.scenario if i.startswith("TechCost")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                i.replace("_", "-").replace("Base", "Baseline").replace("PkBudg", "")
            )
            for i in df.scenario if i.startswith("R2p1")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                i.replace("_", "-").replace("PkBudg", "")
            )
            for i in df.scenario if i.startswith("SusDev")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    "GEI-" + i
                    .replace("_", "-")
                )
            )
            for i in df.scenario if i.startswith("SSP2_")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    i
                    .replace("_", "-")
                )
            )
            for i in df.scenario if i.startswith("LeastTotalCost_")
        ]
    ),
    inplace=True,
)

In [ ]:
#df.filter(scenario="*baseline", keep=False, inplace=True)
#df.filter(scenario="*_LB", keep=False, inplace=True)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    i
                    .replace("full", "Full-CDR")
                    .replace("red", "Reduced-CDR")
                    .replace("eff", "Cost-Effective")
                    .replace("goodpractice", "Good-Practice")
                    .replace("netzero", "NetZero")
                    .replace("1p5C", "1.5°C")
                    .replace("2C", "2°C")
                    .replace("_", "-")
                )
            )
            for i in df.scenario if i.startswith("PEP_")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i, i.replace("COV_", "COVID-Shift-")
            )
            for i in df.scenario
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    i.replace("CO_", "COMMIT-")
                    .replace("CurPol", "Current-Policies")
                    .replace("2Deg", "2°C-")
                    .replace("BAU", "Baseline")
                    .replace("_notax", "-No-Tax")
                    .replace("_2050convergence", "-2050-Convergence")
                )
            )
            for i in df.scenario if i.startswith("CO_")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    "SSP2021-" + i.replace("I_", "-")
                    .replace("D", "Default")
                    .replace("LIRE", "Lifestyle-Renewables")
                    .replace("LI", "Lifestyle")
                    .replace("RE", "Renewables")
                    .replace("LB", "LowBiomass")
                    .replace("_", "-")
                    .replace("-baseline", "-Baseline")
                )
            )
            for i in df.filter(model="IMAGE 3.2").scenario
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    i.replace("CEMICS_", "CEMICS-")
                    .replace("1.5", "1.5°C")
                    .replace("1p5C", "1.5°C")
                    .replace("1p5", "1.5°C")
                    .replace("2.0", "2.0°C")
                    .replace("1p5C", "1.5°C")
                    .replace("1p5", "1.5°C")
                    .replace("2C", "2.0°C")
                    .replace("opt", "Optimal")
                    .replace("_", "-")
                )
            )
            for i in df.scenario if "CEMICS" in i
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    "SDI-" + i.replace("C", "°C").replace("WB", "well-below-")
                )
            )
            for i in df.scenario
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    i.replace("CO_", "COMMIT-")
                    .replace("CurPol", "Current-Policies")
                    .replace("2Deg", "2°C-")
                    .replace("BAU", "Baseline")
                    .replace("_notax", "-No-Tax")
                    .replace("_2050convergence", "-2050-Convergence")
                )
            )
            for i in df.scenario
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    i.replace("R_", "Deep-Mitigation-")
                    .replace("BAU", "Baseline")
                    .replace("SSP_", "Deep-Mitigation-")
                )
            )
            for i in df.scenario
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                    i.replace("CD-LINKS_", "CD-LINKS-")
                    .replace("NoPolicy", "No-Policy")
                )
            )
            for i in df.scenario
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                   i.replace("C", "°C")
                    .replace("WB", "well-below-")
                    .replace("Med", "medium-").replace("_", "-")
                )
            )
            for i in df.scenario if i.startswith("EMF33")
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario=dict(
        [
            (
                i,
                (
                   i.replace("1.5", "1.5°C")
                   .replace("Reference", "Ratchet-Reference")
                   .replace("noOS", "noOvershoot")
                )
            )
            for i in df.filter(model="C-ROADS-5.005").scenario
        ]
    ),
    inplace=True,
)

In [ ]:
df.rename(
    scenario={
        "LowEnergyDemand_1.3_IPCC": "LowEnergyDemand (1.3/IPCC-AR6)",
    },
    inplace=True,
)

In [ ]:
df.region

In [ ]:
df.rename(model={"MESSAGEix-GLOBIOM_1.1": "MESSAGEix-GLOBIOM 1.1"}, inplace=True)

In [ ]:
region_mapping = {
  "Asian countries except Japan": "Asia (R5)",
  "Latin American countries": "Latin America (R5)",
  "Countries of the Middle East and Africa": "Middle East & Africa (R5)",
  "OECD90 and EU (and EU candidate) countries": "OECD & EU (R5)",
  "Countries from the Reforming Economies of the Former Soviet Union": "Reforming Economies (R5)",
  "Countries of Latin America and the Caribbean": "Latin America (R10)",
  "Countries of South Asia; primarily India": "India+ (R10)",
  "Countries of Sub-Saharan Africa": "Africa (R10)",
  "Countries of centrally-planned Asia; primarily China": "China+ (R10)",
  "Countries of the Middle East; Iran, Iraq, Israel, Saudi Arabia, Qatar, etc.": "Middle East (R10)",
  "Eastern and Western Europe (i.e., the EU28)": "Europe (R10)",
  "North America; primarily the United States of America and Canada": "North America (R10)",
  "Pacific OECD": "Pacific OECD (R10)",
  "Reforming Economies of Eastern Europe and the Former Soviet Union; primarily Russia": "Reforming Economies (R10)",
  "Other countries of Asia": "Rest of Asia (R10)",
  "Rest of the World (R10)": "Other (R10)",
  "United States of America": "United States", 
  "Russia": "Russian Federation",
  "European Union (28 member countries)": "European Union and United Kingdom",
  "Rest of the World (R5)": "Other (R5)",
}

In [ ]:
df.rename(region=region_mapping, inplace=True)

In [ ]:
df.filter(region="*(R6)", keep=False, inplace=True)

In [ ]:
df.filter(region=["Colombia", "Pakistan", "Taiwan"], keep=False, inplace=True)

In [ ]:
df.region

In [ ]:
#df = df.filter(region=list(region_mapping.keys()) + ['World'])

In [ ]:
definition = nomenclature.DataStructureDefinition("../../common-definitions/definitions/")

In [ ]:
definition.validate(df)

In [ ]:
# remove variables of little relevance that are not included in common-definitions
df.filter(
    variable=[
        "*AR6 climate diagnostics*",
        "Diagnostics|MAGICC6*",
        "Carbon Sequestration|Other",
        "Secondary Energy",
        "Food Energy Supply",
        "Investment|Energy Supply|Electricity|Non-fossil",
        "Investment|Energy Supply|Extraction|Bioenergy",
        "Investment|Energy Supply|Hydrogen|Renewable",
        "Policy Cost|Consumption Loss",
        "Policy Cost|GDP Loss",
        "Policy Cost|Area under MAC Curve",
        "Price|Agriculture|Non-Energy Crops and Livestock|Index",
        "Policy Cost|Additional Total Energy System Cost",
        "Carbon Sequestration|CCS|Biomass|Energy|Demand|Industry", 
        "Final Energy|Residential and Commercial|Solids|Biomass|Traditional",
        "Final Energy|Transportation|Liquids|Natural Gas",
        "Capacity Additions|Electricity|Storage Capacity",
        "Food Demand",
        "Food Demand|Crops",
        "Food Demand|Livestock",
        "Capacity|Electricity|Peak Demand",
        "Capacity|Electricity|Storage",
        "Secondary Energy|Electricity|Curtailment",
        "Secondary Energy|Electricity|Curtailment|Solar",
        "Secondary Energy|Electricity|Curtailment|Wind",
        "Secondary Energy|Electricity|Storage",
        "Secondary Energy|Electricity|Storage Losses",
        "Secondary Energy|Electricity|Transmission Losses",
    ],
    keep=False,
    inplace=True
)

In [ ]:
# update carbon-management variables
carbon_management_mapping = {
    "Agricultural Demand|Crops|Energy": "Agricultural Demand|Crops|Bioenergy",
    "Agricultural Demand|Crops|Energy|1st generation": "Agricultural Demand|Crops|Bioenergy|1st Generation",
    "Agricultural Demand|Crops|Energy|2nd generation": "Agricultural Demand|Crops|Bioenergy|2nd Generation",
    "Carbon Sequestration|CCS": "Carbon Capture|Geological Storage",
    "Carbon Sequestration|CCS|Biomass": "Carbon Capture|Geological Storage|Biomass",
    "Carbon Sequestration|CCS|Biomass|Energy|Supply": "Carbon Capture|Energy|Supply|Biomass",
    "Carbon Sequestration|CCS|Fossil": "Carbon Capture|Energy|Fossil",
    "Carbon Sequestration|CCS|Fossil|Energy|Demand|Industry": "Carbon Capture|Energy|Demand|Industry",
    "Carbon Sequestration|CCS|Fossil|Energy|Supply": "Carbon Capture|Energy|Supply|Fossil",
    "Carbon Sequestration|CCS|Industrial Processes": "Carbon Capture|Industrial Processes",
    "Carbon Sequestration|Land Use|Afforestation": "Carbon Removal|Land Use|Re/Afforestation",
    "Carbon Sequestration|Direct Air Capture": "Carbon Removal|Geological Storage|Direct Air Capture",
    "Carbon Sequestration|Enhanced Weathering": "Carbon Removal|Enhanced Weathering",
    "Carbon Sequestration|Land Use": "Carbon Removal|Land Use",
    "Yield|Cereal": "Yield|Cropland|Cereals",
    "Yield|Oilcrops": "Yield|Cropland|Oil Crops",
    "Yield|Sugarcrops": "Yield|Cropland|Sugar Crops",
    "Carbon Sequestration|Land Use|Biochar": "Carbon Removal|Land Use|Biochar",
    "Carbon Sequestration|Land Use|Soil Carbon Management": "Carbon Removal|Land Use|Soil Carbon Management",
}

df.rename(variable=carbon_management_mapping, inplace=True)

In [ ]:
project = "engage"
legacy_mapping = {}

for code, attrs in definition.variable.items():
    if project in attrs.extra_attributes:
        legacy_mapping[attrs.__getattr__(project)] = code

df.rename(variable=legacy_mapping, inplace=True)

In [ ]:
# rename units
df.rename(
    unit={
        "US$2010/kW": "USD_2010/kW",
        "billion US$2010/yr": "billion USD_2010/yr",
        "billion US$2010/yr OR local currency/yr": "billion USD_2010/yr",
        "billion US$2010/yr or local currency/yr": "billion USD_2010/yr",
        "US$2010/kW/yr": "USD_2010/kW/yr",
        "US$2010/t CO2": "USD_2010/t CO2",
        "US$2010/tCO2": "USD_2010/t CO2",
        "US$2010/t CO2 or local currency/t CO2": "USD_2010/t CO2",
        "million Ha/yr": "million ha",
        "Million": "million",
        "Mt NOx/yr": "Mt NO2/yr",  
        "Mt N2O/yr": "kt N2O/yr",
        "million m3/yr": "km3/yr",
        "bn m2/yr": "billion m2",
        "bn tkm/yr": "billion tkm/yr",
        "bn pkm/yr": "billion pkm/yr",
        "US$2010/GJ": "USD_2010/GJ",
        "Mt/year": "Mt/yr",
    },
    inplace=True,
)

In [ ]:
df = df.filter(variable="Forestry *", keep=False).append(
    df.filter(variable="Forestry *").rename(unit={"km3/yr": "million m3/yr"})
)

In [ ]:
df

In [ ]:
definition.validate(df)

In [ ]:
df.filter(variable=definition.variable, inplace=True)

In [ ]:
#df.set_meta("Deep-Mitigation (JGCRI)", "Project")
df.set_meta(project_meta_indicator, "Project")

In [ ]:
df.meta

In [ ]:
for model in df.model:
    df.filter(model=model).to_ixmp4(platform)
    print(model)